# Smart Buildings — Official 30-Scenario + 6-Climate Diverse Unseen Evaluation (Run 4 TransferPolicy)

30 official shipped scenarios + 6 diverse unseen scenarios (all 6 climates) x 8640 steps with 4 parallel workers.
Results & leaderboard score land in `/kaggle/working/ablation_results/`.

In [ ]:
import pathlib, subprocess, os

URL  = "https://github.com/KAC-PDW/smart-buildings-competition.git"
REPO = pathlib.Path("/kaggle/working/smart-buildings-competition")
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", URL, str(REPO)], check=True)
os.chdir(REPO)
print(subprocess.run(["git", "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout.strip())

In [ ]:
!pip install -q -e . --no-deps

try:
    import numpy, numba
    print("numpy", numpy.__version__, "| numba ok")
except Exception as exc:
    print("numba broken, pinning numpy 2.0.2:", exc)
    !pip install -q "numpy==2.0.2"

import os
print("cpu count:", os.cpu_count())

In [ ]:
# 288-step official evaluate.py smoke check for TransferPolicy
!python -m smart_buildings.evaluate --policies transfer --buildings building_train_1 --weathers fairbanks_1 --steps 288

In [ ]:
# Official 30-scenario + 6-climate diverse unseen evaluation (36 rollouts x 8640 steps, 4 workers)
!python smart_buildings_toolkit/scripts/run_experiment.py --mode official30 --extra-test-count 6 --workers 4 --skip-install

In [ ]:
!python score_leaderboard.py /kaggle/working/ablation_results/official30_results --policy transfer --expect 30

import json, pandas as pd
df = pd.read_csv("/kaggle/working/ablation_results/ablation_summary.csv")
ship = df[df["split"] == "shipped"].copy()
base = json.loads(open("baseline_returns.json").read())
for b_name in ("pid_water", "fixed_setpoint", "pid"):
    ship[b_name] = [abs(base[f"{r.building}/{r.weather}"][b_name]) / 8640.0 for r in ship.itertuples()]
ship["city"] = ship["weather"].str.rsplit("_", n=1).str[0]
print("\n=== Official 30 Per-City Mean MAE ===")
print(ship.groupby("city")[["mae_vs_zero", "pid_water", "fixed_setpoint", "pid"]].mean().round(5).to_string())
print("\n=== Official 30 Overall Mean MAE ===")
print(ship[["mae_vs_zero", "pid_water", "fixed_setpoint", "pid"]].mean().round(6).to_string())
unseen = df[df["split"] == "unseen_diverse"].copy()
if len(unseen):
    print("\n=== Unseen Diverse (6 Climates) ===")
    print(unseen[["building", "weather", "mae_vs_zero"]].round(5).to_string(index=False))
    print(f"Unseen Diverse Mean MAE: {unseen['mae_vs_zero'].mean():.5f}")